# E2: RanDumb và NCM trên đặc trưng ViT-B/16 (Benchmark F)

Notebook này trích đặc trưng ViT cho 4 dataset × 2 backbone, rồi chạy RanDumb và NCM, mỗi tổ hợp một W&B run với thẻ `lab:reproduction`.

**Chuẩn bị trên Kaggle:**
1. **Settings → Accelerator:** GPU T4 x2. **Internet:** On.
2. **Add-ons → Secrets:** bật `WANDB_API_KEY` cho notebook này. Nên thêm `HF_TOKEN` (token đọc của Hugging Face) để tải nhanh hơn.
3. Ở ô 1, đặt `REPO_BRANCH` là nhánh chứa code E2 (`main` sau khi đã merge).

**Thời gian:** chưa đo. Ước lượng 30-60 phút cho cả 8 lần trích đặc trưng, phần lớn là ViT. Nếu phiên Kaggle bị ngắt, chạy lại ô 5: file đặc trưng đã có sẽ được bỏ qua.

**Dung lượng tải về:** khoảng 5 GB (ImageNet-R khoảng 2,7 GB, CUB khoảng 1 GB, ImageNet-A khoảng 0,8 GB, CIFAR-100 khoảng 0,14 GB).

In [ ]:
# Ô 1. Cấu hình
REPO_URL = "https://github.com/24122039Kien/RanDumb-reproduction.git"
REPO_BRANCH = "main"
REPO_DIR = "/kaggle/working/RanDumb-reproduction"
DATA_DIR = "/kaggle/working/data"

DATASETS = ["cifar100", "cub", "imagenet-r", "imagenet-a"]  # thứ tự ưu tiên
BACKBONES = ["in21k", "in1k"]
METHODS = ["randumb", "ncm"]
NORMALIZE = True          # False: giữ ảnh ở [0, 1] như code gốc
NCM_METRIC = "euclidean"  # bài không nêu; ghi lại lựa chọn
TAGS = ["lab:reproduction", "E2"]

In [ ]:
# Ô 2. Lấy code của nhóm
import os
import subprocess
import sys

if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", REPO_BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, REPO_DIR], check=True)
commit = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("Commit:", commit)
assert os.path.exists(os.path.join(REPO_DIR, "scripts", "run_e2.py")), "Nhánh này chưa có code E2"

In [ ]:
# Ô 3. Secrets và biến môi trường
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")
try:
    os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
    print("Đã có HF_TOKEN")
except Exception:
    print("Không có HF_TOKEN, vẫn chạy được nhưng có thể chậm hơn")
os.environ["RANDUMB_DATA_DIR"] = DATA_DIR
os.environ["PYTHONUNBUFFERED"] = "1"

In [ ]:
# Ô 4. Kiểm tra nhanh: GPU và test của repo
import torch

assert torch.cuda.is_available(), "Chưa bật GPU: Settings → Accelerator → GPU T4 x2"
print(torch.cuda.get_device_name(0))
subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider"], cwd=REPO_DIR, check=True)

In [ ]:
# Ô 5. Trích đặc trưng và chạy E2
import json
import time


def run(script, *args):
    """Chạy một script của repo, in log, trả về JSON in ở cuối (nếu có)."""
    cmd = [sys.executable, os.path.join(REPO_DIR, "scripts", script), *args]
    proc = subprocess.run(cmd, cwd=REPO_DIR, capture_output=True, text=True)
    print(proc.stdout[-1500:])
    if proc.returncode != 0:
        print(proc.stderr[-3000:])
        raise RuntimeError(f"{script} lỗi với {args}")
    text = "\n" + proc.stdout
    start = text.rfind("\n{")
    return json.loads(text[start:]) if start != -1 else None


norm_flag = "--normalize" if NORMALIZE else "--no-normalize"
results = []
for dataset in DATASETS:
    for backbone in BACKBONES:
        t0 = time.time()
        run("extract_vit_features.py", "--dataset", dataset, "--backbone", backbone, norm_flag)
        print(f"== {dataset}/{backbone}: trích đặc trưng {time.time() - t0:.0f} giây")
        for method in METHODS:
            tags = [arg for tag in TAGS for arg in ("--tag", tag)]
            out = run("run_e2.py", "--dataset", dataset, "--backbone", backbone, "--method", method,
                      norm_flag, "--ncm-metric", NCM_METRIC, *tags)
            results.append({"dataset": dataset, "backbone": backbone, "method": method, **out})

In [ ]:
# Ô 6. Bảng kết quả để dán vào README và deck
import pandas as pd

df = pd.DataFrame(results)[["dataset", "backbone", "method", "final_accuracy", "paper_accuracy",
                            "relative_deviation_pct", "wandb_url"]]
pd.set_option("display.max_colwidth", 200)
display(df.round(2))
try:
    print(df.round(2).to_markdown(index=False))
except ImportError:  # thiếu thư viện tabulate
    print(df.round(2).to_csv(index=False))

## Sau khi chạy xong

- [ ] Bấm **Save Version** để giữ output.
- [ ] Gửi bảng ở ô 6 để cập nhật README và `Docs/`.
- [ ] Kiểm tra các link W&B mở được ở cửa sổ ẩn danh.
- [ ] Nếu còn thời gian: đặt `NORMALIZE = False` và chạy lại để so với cách của code gốc.